In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import glob
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:

import os
import glob
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.transforms import InterpolationMode
from PIL import Image

def _list_images(folder):
    exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp")
    files = []
    for e in exts:
        files.extend(glob.glob(os.path.join(folder, e)))
    return sorted(files)

def _stem(p):
    return os.path.splitext(os.path.basename(p))[0]

def _find_dir_with_keywords(root, keywords):
    keywords = [k.lower() for k in keywords]
    for r, d, f in os.walk(root):
        r_low = r.lower()
        if all(k in r_low for k in keywords):
            return r
        for dd in d:
            dd_low = dd.lower()
            if all(k in dd_low for k in keywords):
                return os.path.join(r, dd)
    return None

img_dir = _find_dir_with_keywords(path, ["image"]) or _find_dir_with_keywords(path, ["images"])
msk_dir = _find_dir_with_keywords(path, ["mask"])  or _find_dir_with_keywords(path, ["masks"]) or _find_dir_with_keywords(path, ["label"])

if img_dir is None or msk_dir is None:
    raise ValueError("Could not find image/mask folders inside dataset path.")

img_files = _list_images(img_dir)
msk_files = _list_images(msk_dir)

msk_map = {_stem(p): p for p in msk_files}
pairs = []
for im in img_files:
    s = _stem(im)
    if s in msk_map:
        pairs.append((im, msk_map[s]))

if len(pairs) == 0:
    n = min(len(img_files), len(msk_files))
    pairs = list(zip(img_files[:n], msk_files[:n]))

np.random.seed(42)
np.random.shuffle(pairs)
split_idx = int(0.8 * len(pairs))
train_pairs = pairs[:split_idx]
val_pairs   = pairs[split_idx:]

IMG_SIZE = (256, 256)

img_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
])

mask_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE, interpolation=InterpolationMode.NEAREST),
])

class UNet(Dataset):
    def __init__(self, pairs, img_transform=None, mask_transform=None):
        self.pairs = pairs
        self.img_transform = img_transform
        self.mask_transform = mask_transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, msk_path = self.pairs[idx]

        img = Image.open(img_path).convert("RGB")
        msk = Image.open(msk_path)

        if self.img_transform is not None:
            img = self.img_transform(img)

        if self.mask_transform is not None:
            msk = self.mask_transform(msk)

        msk = np.array(msk)
        if msk.ndim == 3:
            msk = msk[..., 0]
        msk = torch.from_numpy(msk).long()
        msk = remap_mask(msk)

        return img, msk


In [ ]:
train_ds = UNet(train_pairs, img_transform=img_transform, mask_transform=mask_transform)
val_ds   = UNet(val_pairs,   img_transform=img_transform, mask_transform=mask_transform)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np
# Custom Dataset Class
class FloodSegmentationDataset(Dataset):
    def __init__(self, root_dir, csv_file, transform=None, target_transform=None):
        self.root_dir = root_dir
        self.metadata = pd.read_csv(csv_file)
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, "Image", self.metadata.iloc[idx, 0])   # Use the csv to get paths
        mask_path = os.path.join(self.root_dir, "Mask", self.metadata.iloc[idx, 1])

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # Convert mask to grayscale (1 channel = binary segmentation)

        if self.transform:
            image = self.transform(image)

        if self.target_transform:
            mask = self.target_transform(mask)

        # Replace mask values with remapped values
        mask = remap_mask_binary(mask)

        return image, mask       # In image classification datasets, we return image and label. Here, we return image and mask

        from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

# Define transforms for images and masks
image_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((256, 256)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Standard ImageNet normalization
])

mask_transforms = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),  # Keep segmentation masks intact
    transforms.PILToTensor(),

])

## **🔹 Splitting the Dataset into Train & Test**

# Define dataset paths
csv_path = os.path.join(path, "metadata.csv")

# Read metadata CSV
metadata = pd.read_csv(csv_path)

# Split dataset into 80% train, 20% test
train_data, test_data = train_test_split(metadata, test_size=0.2, random_state=42, shuffle=True)

# Save split CSVs in /content
train_path = os.path.join('/content', 'train.csv')
test_path = os.path.join('/content', 'test.csv')

train_data.to_csv(train_path, index=False)
test_data.to_csv(test_path, index=False)

In [ ]:
import segmentation_models_pytorch as smp

# Define U-Net Model
device = "cpu"
model = smp.Unet(
    encoder_name="efficientnet-b0",  # Pretrained encoder (backbone)
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=3,  # RGB images
    classes=1,  # Binary segmentation (1 output channel)
).to(device)

In [ ]:
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        images, masks = images.to(device), masks.to(device).to(torch.float)

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device).to(torch.float)

            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
import torch
from torch import nn
# Define loss function and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

num_epochs = 10  # Define number of epochs
train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")


In [ ]:
# TO DO